# XGBoost 지하철 2호선 승하차 예측 모델

## 개요
- **데이터**: 2024년 서울 지하철 2호선 51개 역 시간대별 승하차 + 기상청 날씨 (366,000행)
- **타깃**: 승차인원, 하차인원 (동시 예측 — MultiOutputRegressor)
- **시계열 분할**: 2024-01 ~ 2024-10 학습 / 2024-11 ~ 2024-12 검증
- **평가지표**: MAE · RMSE · MAPE

### 전처리 완료된 `final_df` 컬럼
| 컬럼 | 설명 |
|------|------|
| 날짜 | datetime |
| 역명 | 2호선 51개 역 |
| 시간 | 5 ~ 23 (정수) |
| 승차인원 / 하차인원 | **타깃** |
| 요일 | 0(월) ~ 6(일) |
| 월 | 1 ~ 12 |
| 공휴일여부 | 0 / 1 |
| 기온 | °C |
| 강수량 / 적설 | mm |


## 1. 라이브러리 로드

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import LabelEncoder
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
import joblib, os, platform

# 한글 폰트
if platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
elif platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

print("라이브러리 로드 완료")


## 2. 데이터 로드

> **전처리 노트북과 같은 커널**에서 실행 중이라면 `final_df`가 이미 메모리에 있습니다.
> 별도 커널에서 실행 시 CSV로 저장 후 읽어오세요.

```python
# 전처리 노트북에서 저장할 때:
# final_df.to_csv("final_df.csv", index=False)
```


In [ ]:
# ── 방법 A: final_df가 메모리에 있는 경우 (같은 커널) ──
# df = final_df.copy()

# ── 방법 B: CSV 파일에서 읽는 경우 ──
# df = pd.read_csv("final_df.csv", parse_dates=["날짜"])

# ── 방법 C: 아래는 테스트용 더미 데이터 (실사용 시 위 방법으로 교체) ──────
import holidays as hd

np.random.seed(42)
dates    = pd.date_range("2024-01-01", "2024-12-31", freq="D")
stations = [
    "시청","을지로입구","을지로3가","을지로4가","동대문역사문화공원(DDP)",
    "신당","상왕십리","왕십리","한양대","뚝섬","성수","건대입구",
    "구의","강변","잠실나루","잠실","잠실새내","종합운동장","삼성",
    "선릉","역삼","강남","교대","서초","방배","사당","낙성대",
    "서울대입구","봉천","신림","신대방","구로디지털단지","대림",
    "신도림","문래","영등포구청","당산","합정","홍대입구","신촌",
    "이대","아현","충정로","디지털미디어시티","합정(2)","당산(2)",
    "영등포구청(2)","문래(2)","신도림(2)","대림(2)","구로디지털단지(2)",
][:51]
hours    = list(range(5, 24))
kr_hols  = hd.KR(years=2024)

rows = []
for date in dates:
    is_hol = int(date.date() in kr_hols)
    for stn in stations:
        for hr in hours:
            peak   = 2.5 if hr in [8, 9, 18, 19] else 1.0
            wkend  = 0.6 if date.weekday() >= 5 else 1.0
            base   = np.random.randint(100, 5000)
            temp   = np.random.uniform(-5, 35)
            rain   = np.random.choice([0.0]*9 + [float(np.random.uniform(1, 30))], 1)[0]
            snow   = np.random.choice([0.0]*19 + [float(np.random.uniform(1, 10))], 1)[0]
            rows.append({
                "날짜":       date,
                "역명":       stn,
                "호선":       "2호선",
                "승차인원":   int(base * peak * wkend * np.random.uniform(0.8, 1.2)),
                "시간":       hr,
                "하차인원":   int(base * peak * wkend * np.random.uniform(0.8, 1.2)),
                "요일":       date.weekday(),
                "월":         date.month,
                "공휴일여부": is_hol,
                "기온":       round(temp, 1),
                "강수량":     round(rain, 1),
                "적설":       round(snow, 1),
            })

df = pd.DataFrame(rows)
print(f"데이터 로드 완료: {len(df):,}행 × {df.shape[1]}열")
df.head()


## 3. 피처 엔지니어링

| 추가 피처 | 이유 |
|-----------|------|
| 시간/월/요일 sin·cos | 순환성 표현 (23시 → 0시 연속성) |
| 출근·퇴근 피크 플래그 | 도메인 지식 명시 주입 |
| 비근무일 | 주말 + 공휴일 통합 |
| 강수·적설 여부 | 희소값(0이 94%)의 이진화 |
| 불쾌지수 | 기온 + 강수 복합 날씨 효과 |
| 역명 LabelEncoding | XGBoost는 범주형 직접 입력 불가 |


In [ ]:
df = df.sort_values(["날짜", "역명", "시간"]).reset_index(drop=True)

# 역명 인코딩
le_station       = LabelEncoder()
df["역명_enc"]   = le_station.fit_transform(df["역명"])

# 시간 주기 인코딩
df["시간_sin"]   = np.sin(2 * np.pi * df["시간"] / 24)
df["시간_cos"]   = np.cos(2 * np.pi * df["시간"] / 24)

# 월 주기 인코딩
df["월_sin"]     = np.sin(2 * np.pi * df["월"] / 12)
df["월_cos"]     = np.cos(2 * np.pi * df["월"] / 12)

# 요일 주기 인코딩
df["요일_sin"]   = np.sin(2 * np.pi * df["요일"] / 7)
df["요일_cos"]   = np.cos(2 * np.pi * df["요일"] / 7)

# 날씨 파생 피처
df["강수_여부"]  = (df["강수량"] > 0).astype(int)
df["적설_여부"]  = (df["적설"]   > 0).astype(int)
df["불쾌지수"]   = (
    9/5 * df["기온"]
    - 0.55 * (1 - df["강수_여부"] * 0.8) * (9/5 * df["기온"] - 26)
    + 32
).round(2)

# 통합 플래그
df["비근무일"]   = ((df["요일"] >= 5) | (df["공휴일여부"] == 1)).astype(int)
df["출근피크"]   = df["시간"].isin([7, 8, 9]).astype(int)
df["퇴근피크"]   = df["시간"].isin([18, 19, 20]).astype(int)

print(f"피처 엔지니어링 완료 — 총 컬럼 수: {df.shape[1]}")
df[["역명","시간","시간_sin","시간_cos","강수_여부","불쾌지수","비근무일","출근피크","퇴근피크"]].head(3)


## 4. 시계열 분할

> **랜덤 분할 금지** — 미래 데이터가 학습에 섞이는 데이터 누수(Data Leakage) 방지
> 날짜 기준 분리: `2024-01 ~ 2024-10` 학습 / `2024-11 ~ 2024-12` 검증


In [ ]:
SPLIT_DATE = "2024-11-01"

train_df = df[df["날짜"] < SPLIT_DATE].copy()
valid_df = df[df["날짜"] >= SPLIT_DATE].copy()

print(f"학습: {len(train_df):>10,}행  ({train_df['날짜'].min().date()} ~ {train_df['날짜'].max().date()})")
print(f"검증: {len(valid_df):>10,}행  ({valid_df['날짜'].min().date()} ~ {valid_df['날짜'].max().date()})")
print(f"분할 비율 — 학습 {len(train_df)/len(df)*100:.1f}% / 검증 {len(valid_df)/len(df)*100:.1f}%")


## 5. 피처 / 타깃 정의

In [ ]:
FEATURE_COLS = [
    "역명_enc",
    "시간",   "시간_sin",  "시간_cos",
    "요일",   "요일_sin",  "요일_cos",
    "월",     "월_sin",    "월_cos",
    "공휴일여부", "비근무일",
    "출근피크", "퇴근피크",
    "기온", "강수량", "적설",
    "강수_여부", "적설_여부", "불쾌지수",
]
TARGET_COLS = ["승차인원", "하차인원"]

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET_COLS]
X_valid = valid_df[FEATURE_COLS]
y_valid = valid_df[TARGET_COLS]

print(f"피처 수 : {len(FEATURE_COLS)}")
print(f"타깃    : {TARGET_COLS}")
print(f"X_train : {X_train.shape}  /  X_valid : {X_valid.shape}")


## 6. XGBoost 모델 정의 및 학습

### 주요 파라미터 설계 근거

| 파라미터 | 값 | 이유 |
|----------|----|------|
| `n_estimators` | 500 | early stopping으로 실제 최적 라운드 자동 결정 |
| `learning_rate` | 0.05 | 낮은 lr + 많은 트리 → 일반화 성능 향상 |
| `max_depth` | 7 | 역·시간·날씨 상호작용 포착 |
| `min_child_weight` | 5 | 공휴일 등 소수 조건 과적합 방지 |
| `reg_alpha` | 0.1 | L1 — 강수량·적설 희소 피처 영향 자동 축소 |
| `subsample` | 0.8 | 행 샘플링으로 과적합 방지 |
| `tree_method` | hist | 36만 행 대용량 고속 학습 |
| `early_stopping_rounds` | 30 | 검증 MAE 30라운드 미개선 시 자동 중단 |


In [ ]:
xgb_base = XGBRegressor(
    n_estimators          = 500,
    learning_rate         = 0.05,
    max_depth             = 7,
    min_child_weight      = 5,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    reg_alpha             = 0.1,
    reg_lambda            = 1.0,
    gamma                 = 0.1,
    tree_method           = "hist",
    random_state          = 42,
    n_jobs                = -1,
    early_stopping_rounds = 30,
    eval_metric           = "mae",
    verbosity             = 0,
)

# MultiOutputRegressor: 승차/하차 독립 모델 2개 병렬 학습
model = MultiOutputRegressor(xgb_base, n_jobs=2)

# 각 내부 estimator에 eval_set 개별 주입
fit_params = {
    f"estimator_{i}__eval_set": [(X_valid, y_valid.iloc[:, i])]
    for i in range(len(TARGET_COLS))
}

print("모델 학습 시작...")
model.fit(X_train, y_train, **fit_params)
print("모델 학습 완료")

for col, est in zip(TARGET_COLS, model.estimators_):
    print(f"  [{col}] best iteration: {est.best_iteration}")


## 7. 예측 (음수 클리핑 포함)

In [ ]:
y_pred = pd.DataFrame(
    np.clip(model.predict(X_valid), 0, None),
    columns=TARGET_COLS,
    index=y_valid.index,
)

# 결과 샘플
sample              = valid_df[["날짜", "역명", "시간"]].copy()
sample["실제_승차"] = y_valid["승차인원"].values
sample["예측_승차"] = y_pred["승차인원"].values.astype(int)
sample["실제_하차"] = y_valid["하차인원"].values
sample["예측_하차"] = y_pred["하차인원"].values.astype(int)
sample.head(10)


## 8. 평가 지표 (MAE / RMSE / MAPE)

In [ ]:
def mape(y_true, y_pred, eps=1e-6):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    mask   = y_true > eps
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

print("=" * 50)
print("  검증 성능  (2024-11 ~ 2024-12)")
print("=" * 50)

results = {}
for col in TARGET_COLS:
    mae      = mean_absolute_error(y_valid[col], y_pred[col])
    rmse     = mean_squared_error(y_valid[col],  y_pred[col], squared=False)
    mape_val = mape(y_valid[col], y_pred[col])
    results[col] = {"MAE": mae, "RMSE": rmse, "MAPE(%)": mape_val}
    print(f"\n  [{col}]")
    print(f"    MAE  : {mae:,.1f} 명")
    print(f"    RMSE : {rmse:,.1f} 명")
    print(f"    MAPE : {mape_val:.2f} %")

print("=" * 50)


## 9. 피처 중요도 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, estimator, label in zip(axes, model.estimators_, TARGET_COLS):
    imp        = estimator.feature_importances_
    sorted_idx = np.argsort(imp)[::-1]
    top_n      = 15

    ax.barh(
        [FEATURE_COLS[i] for i in sorted_idx[:top_n]][::-1],
        imp[sorted_idx[:top_n]][::-1],
        color="#7F77DD", edgecolor="none",
    )
    ax.set_title(f"피처 중요도 — {label}", fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("Importance (gain)", fontsize=11)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="y", labelsize=10)

plt.suptitle("XGBoost 피처 중요도 (Top 15)", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()


## 10. 예측값 vs 실제값 시각화

In [ ]:
TARGET_STATION = "강남"
if TARGET_STATION not in le_station.classes_:
    TARGET_STATION = le_station.classes_[0]

stn_enc   = le_station.transform([TARGET_STATION])[0]
mask      = valid_df["역명_enc"] == stn_enc
DAYS_SHOW = 7
cut       = min(mask.sum(), DAYS_SHOW * 19)

fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)

for ax, col in zip(axes, TARGET_COLS):
    actual = y_valid.loc[mask, col].values[:cut]
    pred   = y_pred.loc[mask,  col].values[:cut]

    ax.plot(actual, label="실제", color="#3C3489", linewidth=1.8)
    ax.plot(pred,   label="예측", color="#EF9F27", linewidth=1.5, linestyle="--")
    ax.fill_between(range(len(actual)), actual, pred, alpha=0.12, color="#7F77DD")
    ax.set_title(f"{TARGET_STATION}역  {col} — 예측 vs 실제 (검증 첫 {DAYS_SHOW}일)", fontsize=12)
    ax.set_ylabel("인원 수", fontsize=10)
    ax.legend(fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", linewidth=0.5, alpha=0.4)

axes[-1].set_xlabel("시간대 인덱스 (검증 기간)", fontsize=10)
plt.tight_layout()
plt.savefig(f"prediction_{TARGET_STATION}.png", dpi=150, bbox_inches="tight")
plt.show()


## 11. 역별 MAPE 비교 (승차인원 기준)

In [ ]:
station_mapes = []
for stn in le_station.classes_:
    stn_enc = le_station.transform([stn])[0]
    m       = valid_df["역명_enc"] == stn_enc
    if m.sum() == 0:
        continue
    m_val = mape(y_valid.loc[m, "승차인원"], y_pred.loc[m, "승차인원"])
    station_mapes.append({"역명": stn, "MAPE(%)": round(m_val, 2)})

mape_df = pd.DataFrame(station_mapes).sort_values("MAPE(%)")
print("MAPE 낮은 역 TOP 10 (예측 정확도 높음)")
display(mape_df.head(10))
print("\nMAPE 높은 역 TOP 10 (예측 어려운 역)")
display(mape_df.tail(10))

fig, ax = plt.subplots(figsize=(14, 6))
colors  = ["#7F77DD" if v < mape_df["MAPE(%)"].median() else "#F0997B"
           for v in mape_df["MAPE(%)"]]
ax.bar(mape_df["역명"], mape_df["MAPE(%)"], color=colors, edgecolor="none")
ax.axhline(mape_df["MAPE(%)"].median(), color="#3C3489", linestyle="--",
           linewidth=1.2, label=f"중앙값 {mape_df['MAPE(%)'].median():.1f}%")
ax.set_title("역별 승차인원 예측 MAPE (%)", fontsize=13)
ax.set_ylabel("MAPE (%)", fontsize=11)
ax.set_xticklabels(mape_df["역명"], rotation=60, ha="right", fontsize=9)
ax.legend(fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("station_mape.png", dpi=150, bbox_inches="tight")
plt.show()


## 12. 모델 저장

In [ ]:
os.makedirs("models", exist_ok=True)
joblib.dump(model,      "models/xgb_subway_model.pkl")
joblib.dump(le_station, "models/label_encoder_station.pkl")

print("저장 완료")
print("  models/xgb_subway_model.pkl")
print("  models/label_encoder_station.pkl")


## 13. 단일 예측 추론 함수

In [ ]:
import holidays as hd

def predict_single(
    station: str,
    date_str: str,
    hour: int,
    temp: float,
    rain: float = 0.0,
    snow: float = 0.0,
) -> dict:
    """
    단일 조건 승차/하차 예측.

    Parameters
    ----------
    station  : 역명 (예: '강남')
    date_str : 날짜 문자열 (예: '2025-01-10')
    hour     : 시간 (5 ~ 23)
    temp     : 기온 (°C)
    rain     : 강수량 (mm, 기본 0)
    snow     : 적설량 (mm, 기본 0)
    """
    date    = pd.Timestamp(date_str)
    kr_hols = hd.KR(years=date.year)
    is_hol  = int(date.date() in kr_hols)
    stn_enc = le_station.transform([station])[0]

    row = {
        "역명_enc":   stn_enc,
        "시간":       hour,
        "시간_sin":   np.sin(2 * np.pi * hour / 24),
        "시간_cos":   np.cos(2 * np.pi * hour / 24),
        "요일":       date.weekday(),
        "요일_sin":   np.sin(2 * np.pi * date.weekday() / 7),
        "요일_cos":   np.cos(2 * np.pi * date.weekday() / 7),
        "월":         date.month,
        "월_sin":     np.sin(2 * np.pi * date.month / 12),
        "월_cos":     np.cos(2 * np.pi * date.month / 12),
        "공휴일여부": is_hol,
        "비근무일":   int(date.weekday() >= 5 or bool(is_hol)),
        "출근피크":   int(hour in [7, 8, 9]),
        "퇴근피크":   int(hour in [18, 19, 20]),
        "기온":       temp,
        "강수량":     rain,
        "적설":       snow,
        "강수_여부":  int(rain > 0),
        "적설_여부":  int(snow > 0),
        "불쾌지수":   round(9/5*temp - 0.55*(1-int(rain>0)*0.8)*(9/5*temp-26)+32, 2),
    }
    X    = pd.DataFrame([row])[FEATURE_COLS]
    pred = np.clip(model.predict(X), 0, None)[0]
    return {
        "역명":          station,
        "날짜":          date_str,
        "시간":          f"{hour}시",
        "예측_승차인원": int(pred[0]),
        "예측_하차인원": int(pred[1]),
    }


# ── 사용 예시 ──
examples = [
    ("강남", "2025-01-10",  9,  3.0, 0.0, 0.0),   # 평일 출근 피크, 맑음
    ("강남", "2025-01-10", 18,  3.0, 0.0, 0.0),   # 평일 퇴근 피크
    ("강남", "2025-01-11", 14,  1.5, 5.2, 0.0),   # 토요일, 비
    ("강남", "2025-01-01", 12, -2.0, 0.0, 3.0),   # 신정 공휴일, 적설
]

print(f"{'역명':<8} {'날짜':<12} {'시간':<6} {'예측승차':>8} {'예측하차':>8}")
print("-" * 50)
for args in examples:
    r = predict_single(*args)
    print(f"{r['역명']:<8} {r['날짜']:<12} {r['시간']:<6} {r['예측_승차인원']:>8,} {r['예측_하차인원']:>8,}")
